**Author**: JW Debelius (justine.debelius@jhu.edu)<br>
**Project**: 017-Arg<br>
**Enviroment**: <br>
**Creation Date**: July 2021<br>
**Update Date**: November 2025


Here is a list of the most interesting bacteria to investigate the abundance of:
- E. coli 

- Klebsiella

- Enterobacter cloace

- Pseudomonas

- Streptococci  

- Enterococci 

- staphylococci

- Clostridioides (difficile/perfringens)

- Bactetoides (fragilis)

Enterobacterales - E.coli, Klebisella and Enterobacter is all bacteria in the Enterobacterales family. If it is difficult to investigate the abundance of the bacteria, the abundance of Enterobacterales is very interesting as well. The abundance of E.coli and Klebsiella is most important. 

 

I went to the GTDB website and downloaded files with all the possible anntoations for each of these clades so we cna try and find any feature that might map. So, I'm going tor ead in the Klebsiella as an example and we can m aybe fo from htere.

In [1]:
import itertools as it
import os
import re

import biom
from matplotlib import rcParams
import matplotlib.pyplot as plt
from matplotlib import ticker

import numpy as np
import pandas as pd
import scipy
import seaborn as sn
import skbio
import patsy
import statsmodels.formula.api as smf
import statsmodels.api as sms

from qiime2 import Artifact, Metadata, Visualization

rcParams['font.sans-serif'] = ['Helvetica', 'Arial']
rcParams['pdf.fonttype'] = 42
np.set_printoptions(precision=5)

%matplotlib inline

In [2]:
abx_order = ['untreated', 'nitrofurantoin', 'pivmecillinam',
             'co-trimoxazole', 'ceftibuten', 'ciprofloxacin']
colors = ['#969696', '#009E73','#9D654C', 
          '#4E7705', '#098BD9',  '#CC79A7']
spectrum_order = ['untreated', 'narrow', 'broad']
alpha_metrics = ['observed_features', 'shannon', 'simpson']
abx_names = {str(i): abx for i, abx in enumerate(abx_order)}
abx_colors = {str(i): color for i, color in enumerate(colors)}
abx_colors2 = {i: color for i, color in enumerate(colors)}

# Data

## Metadata

In [3]:
meta_all = pd.read_csv('final-data/01.metadata/analysis_sample_meta.tsv',
                     sep='\t',
                     dtype=str)
meta_all.set_index('sample-id', inplace=True)
meta_all[['age', 'days_since_intervention', 'gtdb_depth']] = \
    meta_all[['age', 'days_since_intervention', 'gtdb_depth']].astype(float)

meta = meta_all[['host_subject_id', 'timepoint', 'days_since_intervention',
             'abx_code2', 'age', 'sex', 'vegetarian', 'extraction_no', 'after_travel_na_eu']].copy()

## Table

In [4]:
table_q2 = Artifact.load('final-data/02.tables/matched-species-gtdb.qza')
table = table_q2.view(pd.DataFrame)

/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_demux/_summarize/_visualizer.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Reference taxa

In [5]:
total_counts = table.sum(axis=1)

In [6]:
prevotella = table[[x for x in table.columns if 's__Prevotella ' in x]].copy()

## Klebsiella

In [7]:
klebsiella = pd.read_csv('reference/gtdb_taxa/klebsiella.txt', sep='\t', dtype=str)
klebsiella = klebsiella.loc[~klebsiella['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
keep_kleb = klebsiella['gtdb_taxonomy'].apply(lambda x: ('g__Klebsiella' in x) | ('g__Kosakonia' in x) | ('g__Kluyvera' in x))
klebsiella_counts = table[[x for x in table.columns if ('g__Klebsiella' in x) | ('g__Kosakonia' in x) | ('g__Kluyvera' in x)]].copy()

In [8]:
klebsiella_n = klebsiella_counts.div(total_counts, axis=0)
klebsiella_n.sum(axis=1).describe()

count    4.730000e+02
mean     8.275475e-05
std      5.563526e-04
min      0.000000e+00
25%      2.650806e-07
50%      1.428840e-06
75%      9.362726e-06
max      8.568133e-03
dtype: float64

In [9]:
((klebsiella_n.sum(axis=1) < 1e-5) * 1).mean()

0.7526427061310782

In [10]:
klebsiella_alr = np.log2((klebsiella_counts.sum(axis=1) + 1) / (prevotella.sum(axis=1) + 1))

## E. coli/Shigella

In [11]:
e_coli = pd.concat(axis=0, objs=[
    pd.read_csv('reference/gtdb_taxa/e_coli.txt', sep='\t', dtype=str),
    pd.read_csv('reference/gtdb_taxa/shigella.txt', sep='\t', dtype=str),
])
e_coli.drop_duplicates(inplace=True)
e_coli.reset_index(drop=True, inplace=True)
e_coli = e_coli.loc[~e_coli['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
ecoli_counts = table[[x for x in table.columns if ('g__Escherichia' in x)]].copy()

In [12]:
ecoli_n = ecoli_counts.div(total_counts, axis=0)
ecoli_n.sum(axis=1).describe()

count    473.000000
mean       0.000699
std        0.002447
min        0.000003
25%        0.000014
50%        0.000057
75%        0.000349
max        0.028957
dtype: float64

In [13]:
(ecoli_n.sum(axis=1) < 1e-5).mean()

0.1733615221987315

In [14]:
ecoli_alr = np.log2((ecoli_counts.sum(axis=1) + 1) / (prevotella.sum(axis=1) + 1))

## Enterobacter cloacae

In [15]:
ecoloa = pd.read_csv('reference/gtdb_taxa/e_cloacae.txt', sep='\t', dtype=str)
ecoloa = ecoloa.loc[~ecoloa['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
e_cloacae = table[[x for x in table.columns if 's__Enterobacter cloa' in x]].copy()

In [16]:
e_cloc_n = e_cloacae.div(total_counts, axis=0)
e_cloc_n.sum(axis=1).describe()

count    4.730000e+02
mean     4.423392e-06
std      2.805836e-05
min      0.000000e+00
25%      0.000000e+00
50%      7.244298e-08
75%      5.862742e-07
max      5.335636e-04
dtype: float64

In [17]:
(e_cloc_n.sum(axis=1) > 1e-5).mean()

0.07399577167019028

In [18]:
e_cloacae_alr = np.log2((e_cloacae.sum(axis=1) + 1) / (prevotella.sum(axis=1) + 1))

## Pseudomonas

In [19]:
pmonas = pd.read_csv('reference/gtdb_taxa/pseudomonas.txt', sep='\t', dtype=str)
pmonas = pmonas.loc[~pmonas['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
pseudomonas = table[[x for x in table.columns if ('g__Pseudomonas' in x) & ('g__Pseudomonas_S' not in x)]].copy()

In [20]:
pseudomonas_n = pseudomonas.div(total_counts, axis=0)
pseudomonas_n.sum(axis=1).describe()

count    473.000000
mean       0.000078
std        0.000137
min        0.000003
25%        0.000020
50%        0.000037
75%        0.000083
max        0.001633
dtype: float64

In [21]:
(pseudomonas_n.sum(axis=1) > 1e-5).mean()

0.9534883720930233

In [22]:
pseudomonas_alr = np.log2((pseudomonas.sum(axis=1) + 1) / (prevotella.sum(axis=1) + 1))

## Staphylococcus

In [23]:
staph = pd.read_csv('reference/gtdb_taxa/streptococcus.txt', sep='\t', dtype=str)
staph = staph.loc[~staph['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
staphylococcus = table[[x for x in table.columns if ('g__Staphylococcus' in x)]].copy()

In [24]:
staph_n = staphylococcus.div(total_counts, axis=0)
staph_n.sum(axis=1).describe()

count    4.730000e+02
mean     3.907206e-06
std      8.499452e-06
min      3.138596e-08
25%      1.016203e-06
50%      1.791017e-06
75%      3.616975e-06
max      8.885357e-05
dtype: float64

In [25]:
(staph_n.sum(axis=1) > 1e-5).mean()

0.05708245243128964

## Streptococcus

In [26]:
strep = pd.read_csv('reference/gtdb_taxa/strep2.txt', sep='\t', dtype=str)
strep = strep.loc[~strep['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
streptococcus = table[[x for x in table.columns if ('g__Streptococcus' in x)]].copy()

In [27]:
strep_n = streptococcus.div(total_counts, axis=0)
strep_n.sum(axis=1).describe()

count    473.000000
mean       0.000633
std        0.000953
min        0.000018
25%        0.000146
50%        0.000339
75%        0.000765
max        0.013667
dtype: float64

In [28]:
(strep_n.sum(axis=1) > 1e-5).mean()

1.0

## Clostridium dificle

In [29]:
cdiff = pd.read_csv('reference/gtdb_taxa/c_difficile.txt', sep='\t', dtype=str)
cdiff = cdiff.loc[~cdiff['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
cdiff = table[[x for x in table.columns if ' s__Clostridioides difficile' in x]].copy()
cdiff.shape

(473, 0)

## C perfringes

In [30]:
cper = pd.read_csv('reference/gtdb_taxa/c_perfringens.txt', sep='\t', dtype=str)
cper = cper.loc[~cper['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
s_perfringes = table[[x for x in table.columns if ' s__Sarcina perfringens' in x]].copy()
s_perfringes.shape

(473, 0)

# B fragillis

In [31]:
bfrag = pd.read_csv('reference/gtdb_taxa/b_frag.txt', sep='\t', dtype=str)
bfrag = bfrag.loc[~bfrag['gtdb_taxonomy'].isin(['Undefined (Failed Quality Check)'])].copy()
b_fragilis = table[[x for x in table.columns if 's__Bacteroides fragilis' in x]].copy()
b_fragilis.shape

(473, 2)

In [32]:
bfrag_n = b_fragilis.div(total_counts, axis=0)
bfrag_n.sum(axis=1).describe()

count    473.000000
mean       0.004815
std        0.014364
min        0.000005
25%        0.000248
50%        0.000502
75%        0.002892
max        0.135211
dtype: float64

In [33]:
(bfrag_n.sum(axis=1) > 1e-5).mean()

0.9957716701902748

## Combined data

In [34]:
mapped = pd.DataFrame.from_dict(
    data={
        'klebsiella': klebsiella_counts.sum(axis=1), 
        'e_coli': ecoli_counts.sum(axis=1),
        'e_cloacae': e_cloacae.sum(axis=1),
        'pseudomonas': pseudomonas.sum(axis=1),
        'streptococcus': streptococcus.sum(axis=1),
        'staphylococcus': staphylococcus.sum(axis=1),
        'c_diff': prevotella.sum(axis=1) * 0,
        'c_perfringes': prevotella.sum(axis=1) * 0,
        'b_fragilis': b_fragilis.sum(axis=1),
    },
    orient='columns')
# mapped['subject_id'] = sample_meta['host_subject_id']
# mapped['timepoint'] = sample_meta['timepoint']
# mapped.set_index(['subject_id', 'timepoint'], append=True, inplace=True)

In [35]:
mapped_n = mapped.div(total_counts, axis=0)
mapped_n['host_subject_id'] = meta['host_subject_id']
mapped_n.set_index('host_subject_id', inplace=True)
mapped_perc_count = \
    (mapped_n > 1e-5).groupby('host_subject_id').sum()
mapped_perc_count = (mapped_perc_count > 2).mean() > 0.1

In [36]:
mapped_perc_count.loc[~mapped_perc_count]

e_cloacae         False
staphylococcus    False
c_diff            False
c_perfringes      False
dtype: bool

In [37]:
map_alr = mapped[mapped_perc_count.index[mapped_perc_count]]
map_alr = np.log2((map_alr + 1).div(prevotella.sum(axis=1) + 1, axis=0))
map_alr['subject_id'] = meta['host_subject_id']
map_alr['extraction_no'] = meta['extraction_no']
map_alr['travel_outside_na_eu'] = meta['after_travel_na_eu']
map_alr['timepoint'] = meta['timepoint']
map_alr['days_inv'] = 30 / meta['days_since_intervention']
map_alr['abx_code2'] = meta['abx_code2']
map_alr['period'] = (meta['timepoint'].astype(float) >= 1) * 1 + \
    (meta['timepoint'].astype(float) >= 3) * 1

In [38]:
period_fits = {
    org: smf.mixedlm(f'{org} ~ C(period)*C(abx_code2) + C(extraction_no) + C(travel_outside_na_eu)',
                      data=map_alr,
                      groups=map_alr['subject_id'],
                      ).fit()
    for org in mapped_perc_count.index[mapped_perc_count]
}

In [39]:
def extract_params(clade, fit):
    """
    
    """
    summary = pd.DataFrame.from_dict(
        orient='columns',
        data={'param': fit.params,
              'bse': fit.bse,
              't': fit.tvalues,
              'p': fit.pvalues,
              'ci_lo': fit.conf_int()[0],
              'ci_hi': fit.conf_int()[1],
              },
    )
    summary['clade'] = clade
    summary.drop(index=[i for i in summary.index 
                        if not (('period' in i) | ('abx_code2' in i))],
                 inplace=True)
    def _find_period(x):
        if 'period' not in x:
            return '0'
        else:
            span_ = re.search('C\(period\)\[T.', x).span()[1]
            return x[span_]
    def _find_abx(x):
        if 'abx_code2' not in x:
            return '0'
        else:
            span_ = re.search('C\(abx_code2\)\[T.', x).span()[1]
            return x[span_]
    
    summary['period'] = summary.index.to_frame()[0].apply(_find_period)
    summary['abx'] = summary.index.to_frame()[0].apply(_find_abx)

    return summary

In [40]:
clade_params = pd.concat(
    axis=0,
    objs=[extract_params(clade, fit) for clade, fit in period_fits.items()]
)
clade_params.set_index(['clade', 'abx', 'period'], inplace=True)
clade_params.sort_index(ascending=True, inplace=True)
fdr_q = clade_params['p'].unstack(0).apply(
    lambda x: pd.Series(sms.stats.multipletests(x)[1], index=x.index), 
    axis=1)
clade_params['q'] = fdr_q.unstack([0, 1])

In [41]:
period_fits2 = {
    org: smf.mixedlm(f'{org} ~ C(abx_code2)*days_inv + C(extraction_no) + C(travel_outside_na_eu)',
                      data=map_alr.loc[map_alr['timepoint'].astype(float) > 0],
                      groups=map_alr.loc[map_alr['timepoint'].astype(float) > 0, 'subject_id'],
                      ).fit()
    for org in mapped_perc_count.index[mapped_perc_count]
}

In [42]:
def extract_params2(clade, fit):
    """
    
    """
    summary = pd.DataFrame.from_dict(
        orient='columns',
        data={'param': fit.params,
              'bse': fit.bse,
              't': fit.tvalues,
              'p': fit.pvalues,
              'ci_lo': fit.conf_int()[0],
              'ci_hi': fit.conf_int()[1],
              },
    )
    summary['clade'] = clade
    summary.drop(index=[i for i in summary.index 
                        if not (('days_inv' in i) | ('abx_code2' in i))],
                 inplace=True)
    def _find_abx(x):
        if 'abx_code2' not in x:
            return '0'
        else:
            span_ = re.search('C\(abx_code2\)\[T.', x).span()[1]
            return x[span_]
    
    summary['period'] = summary.index.to_frame()[0].apply(lambda x: 'days_inv' in x)
    summary['abx'] = summary.index.to_frame()[0].apply(_find_abx)

    return summary

In [43]:
clade_params2 = pd.concat(
    axis=0,
    objs=[extract_params2(clade, fit) for clade, fit in period_fits2.items()],
)
clade_params2.set_index(['clade', 'abx', 'period'], inplace=True)
clade_params2.sort_index(ascending=True, inplace=True)
fdr_q2 = clade_params2['p'].unstack(0).apply(
    lambda x: pd.Series(sms.stats.multipletests(x)[1], index=x.index), 
    axis=1)
clade_params2['q'] = fdr_q2.unstack([0, 1])

In [44]:
# clade_params2

In [45]:
clade_params2['a_tidy'] = clade_params2.apply(
    lambda x: '{param:1.2f} ({ci_lo:1.2f}, {ci_hi:1.2f})'.format(**x), 
    axis=1)
clade_params2['a_tidy'] = clade_params2['a_tidy'].mask(
    clade_params2.index.to_frame()['abx'].isin(['0', 0]), '0 (ref)')
clade_params2['q-value'] = clade_params2['q'].apply(
    lambda x: f'{x:1.0e}' if x < 0.01 else f'{x:1.2f}')
clade_params2['q-value'] = clade_params2['q-value'].mask(
    clade_params2.index.to_frame()['abx'].isin(['0', 0]), '')
clade_params2.columns.set_names('stat', inplace=True)
clade_tidy = clade_params2[['a_tidy', 'q-value']].copy().reset_index()
clade_first = clade_tidy.groupby(['clade', 'period']).first()
clade_first = clade_first.mask(np.ones(clade_first.shape) == 1, '')
clade_first.reset_index(inplace=True)
clade_tidy = pd.concat(axis=0, objs=[clade_first, clade_tidy])
# clade_tidy.rename(
#     index={'b_fragilis': 'Bacteriodetes fragilis',
#            'e_coli': 'genus Escherichia',
#           },
#     inplace=True,
#     )
# clade_tidy.rename({i: f'genus {i}'.title().replace('G', 'g') for i in clade_tidy.index if (' ' not in i)},
#                   inplace=True)
# clade_tidy.rename(columns={str(i): abx for i, abx in  enumerate(abx_order)},
#                   level='abx', 
#                   inplace=True)
# clade_tidy.rename(columns={'a_tidy': 'ALR (95% CI)',
#                            'q-value': 'FDR p-value'},
#                   level='stat', 
#                   inplace=True)
# clade_tidy.rename(columns={False: 'New baseline (unitless)',
#                            True: 'Recovery (30 days)$^{-1}$',
#                           },
#                   level='period',
#                   inplace=True,
#                   )
clade_tidy2 = \
    clade_tidy.pivot(index=['clade', 'abx'],
                     columns='period',
                     values=['a_tidy', 'q-value']
                    )
clade_tidy2.columns.set_names(['stat', 'slope'], inplace=True)
clade_tidy2.columns = clade_tidy2.columns.reorder_levels(['slope', 'stat'])
clade_tidy2.replace({(False, 'a_tidy') : {np.nan: '0 (ref)'},
                     (False, 'q-value') : {np.nan: ''}
                    },
                    inplace=True)
clade_tidy2.sort_index(axis='columns', ascending=[False, True], inplace=True)
clade_tidy2.rename(columns={True: 'ALR(30 days)-1', False: 'ALR'},
                    inplace=True)
fixed_names = {'b_fragilis': 'Bacteriodetes fragilis',
               'e_coli': 'genus Escherichia',
              }
clade_tidy2.rename(
    index={name: fixed_names.get(name, 'g. {0}'.format(name.title()))
           for name in clade_first['clade'].unique()}, 
    level='clade',
    inplace=True,
    )
clade_tidy2.rename(index=abx_names, level='abx', inplace=True)
clade_tidy2

slope                                        ALR(30 days)-1          \
stat                                                 a_tidy q-value   
clade                  abx                                            
Bacteriodetes fragilis                                                
                       untreated                    0 (ref)           
                       nitrofurantoin   -0.01 (-0.52, 0.50)    0.98   
                       pivmecillinam    -0.49 (-1.04, 0.07)    0.36   
                       co-trimoxazole  -0.52 (-0.98, -0.05)    0.14   
                       ceftibuten       -0.35 (-0.84, 0.14)    0.51   
                       ciprofloxacin    -0.30 (-0.78, 0.17)    0.51   
genus Escherichia                                                     
                       untreated                    0 (ref)           
                       nitrofurantoin   -0.10 (-0.79, 0.59)    0.95   
                       pivmecillinam    -0.26 (-1.01, 0.48)    0.93   
                       co-trimoxazole  -0.62 (-1.25, -0.00)    0.18   
                       ceftibuten      -0.68 (-1.34, -0.02)    0.20   
                       ciprofloxacin    -0.34 (-0.98, 0.30)    0.51   
g. Klebsiella                                                         
                       untreated                    0 (ref)           
                       nitrofurantoin    0.36 (-0.49, 1.21)    0.87   
                       pivmecillinam    -0.05 (-0.98, 0.87)    0.93   
                       co-trimoxazole   -0.45 (-1.22, 0.33)    0.59   
                       ceftibuten       -0.55 (-1.37, 0.27)    0.51   
                       ciprofloxacin    -0.22 (-1.01, 0.57)    0.59   
g. Pseudomonas                                                        
                       untreated                    0 (ref)           
                       nitrofurantoin    0.19 (-0.35, 0.73)    0.87   
                       pivmecillinam    -0.20 (-0.79, 0.39)    0.93   
                       co-trimoxazole   -0.15 (-0.65, 0.34)    0.80   
                       ceftibuten       -0.28 (-0.81, 0.24)    0.51   
                       ciprofloxacin      0.58 (0.07, 1.09)    0.12   
g. Streptococcus                                                      
                       untreated                    0 (ref)           
                       nitrofurantoin     0.51 (0.04, 0.98)    0.16   
                       pivmecillinam     0.10 (-0.41, 0.61)    0.93   
                       co-trimoxazole    0.04 (-0.38, 0.47)    0.84   
                       ceftibuten       -0.02 (-0.47, 0.44)    0.94   
                       ciprofloxacin     0.38 (-0.06, 0.82)    0.30   

slope                                                   ALR          
stat                                                 a_tidy q-value  
clade                  abx                                           
Bacteriodetes fragilis                                               
                       untreated                    0 (ref)          
                       nitrofurantoin   -2.97 (-6.45, 0.50)    0.39  
                       pivmecillinam    -3.25 (-6.87, 0.36)    0.33  
                       co-trimoxazole  -4.19 (-7.55, -0.84)    0.07  
                       ceftibuten       -2.71 (-6.17, 0.76)    0.42  
                       ciprofloxacin    -2.27 (-5.77, 1.22)    0.68  
genus Escherichia                                                    
                       untreated                    0 (ref)          
                       nitrofurantoin   -1.31 (-4.74, 2.11)    0.59  
                       pivmecillinam    -1.28 (-4.86, 2.30)    0.93  
                       co-trimoxazole   -2.10 (-5.41, 1.20)    0.38  
                       ceftibuten       -2.30 (-5.72, 1.12)    0.46  
                       ciprofloxacin    -0.68 (-4.11, 2.75)    0.88  
g. Klebsiella                                                        
                       untreated         

In [47]:
clade_tidy2.to_clipboard()